# Coding Fundamentals for Agentic AI Infrastructure

This notebook covers the core engineering skills needed to build, test, and maintain production agentic AI systems.

## Topics
1. **Python for AI Agents** — async/await, type hints, dataclasses, context managers, generators
2. **Bash Scripting for AI Workflows** — automation, piping, process management
3. **TypeScript Patterns for AI Apps** — types, interfaces, async patterns
4. **Data Structures & Algorithms for Agents** — queues, graphs, priority queues
5. **Git & Version Control for AI Projects** — branching, model versioning
6. **Testing with PyTest** — mocking LLM calls, testing agent functions


---
## 1. Python for AI Agents

Modern AI agents are inherently concurrent — they call multiple LLMs, tools, and APIs simultaneously. Python's `asyncio` ecosystem is the backbone of this concurrency model.

### 1.1 Async / Await

Use `async def` to define coroutines. Use `await` to yield control while waiting for I/O (e.g., an LLM API call).
Use `asyncio.gather` to run multiple coroutines concurrently.


In [ ]:
import asyncio
import time

# Simulate an LLM API call with a delay
async def call_llm(prompt: str, model: str = "gpt-4o", delay: float = 1.0) -> str:
    """Simulated async LLM call."""
    await asyncio.sleep(delay)  # non-blocking wait
    return f"[{model}] Response to: '{prompt}'"

async def sequential_calls():
    start = time.perf_counter()
    r1 = await call_llm("What is RAG?", delay=1.0)
    r2 = await call_llm("What is an agent?", delay=1.0)
    elapsed = time.perf_counter() - start
    print(f"Sequential: {elapsed:.2f}s")
    print(r1)
    print(r2)

async def concurrent_calls():
    start = time.perf_counter()
    r1, r2 = await asyncio.gather(
        call_llm("What is RAG?", delay=1.0),
        call_llm("What is an agent?", delay=1.0),
    )
    elapsed = time.perf_counter() - start
    print(f"Concurrent: {elapsed:.2f}s")
    print(r1)
    print(r2)

# In a notebook we use await directly (Jupyter runs an event loop)
await sequential_calls()
print()
await concurrent_calls()

In [ ]:
import asyncio

# Timeouts and error handling in async agent calls
async def call_with_timeout(prompt: str, timeout: float = 2.0) -> str:
    try:
        result = await asyncio.wait_for(
            call_llm(prompt, delay=3.0),  # will exceed timeout
            timeout=timeout
        )
        return result
    except asyncio.TimeoutError:
        return f"TIMEOUT: LLM call exceeded {timeout}s"

# Retry with exponential backoff
async def call_with_retry(prompt: str, max_retries: int = 3) -> str:
    for attempt in range(max_retries):
        try:
            return await asyncio.wait_for(call_llm(prompt, delay=0.1), timeout=1.0)
        except asyncio.TimeoutError:
            wait = 2 ** attempt
            print(f"Attempt {attempt+1} failed. Retrying in {wait}s...")
            await asyncio.sleep(wait)
    return "FAILED after all retries"

result = await call_with_timeout("Hello", timeout=2.0)
print(result)

result2 = await call_with_retry("Hello")
print(result2)

### 1.2 Type Hints

Type hints make agent codebases self-documenting and catch bugs early with tools like `mypy` or `pyright`.
They are especially important when defining message schemas, tool signatures, and agent state.


In [ ]:
from typing import Literal, TypedDict, Optional, Union, Callable, Awaitable

# Message roles common in chat APIs
Role = Literal["system", "user", "assistant", "tool"]

class Message(TypedDict):
    role: Role
    content: str

class ToolCall(TypedDict):
    id: str
    name: str
    arguments: dict

# Type alias for an agent tool function
ToolFunction = Callable[[dict], Awaitable[str]]

# Generic agent response that can carry tool calls or text
AgentOutput = Union[str, list[ToolCall]]

def build_message(role: Role, content: str) -> Message:
    return {"role": role, "content": content}

history: list[Message] = [
    build_message("system", "You are a helpful assistant."),
    build_message("user", "What tools do you have?"),
]

for msg in history:
    print(f"{msg['role'].upper():>10}: {msg['content']}")

### 1.3 Dataclasses

Dataclasses provide a clean, type-safe way to define agent state, tool definitions, and configuration objects — without the boilerplate of `__init__`, `__repr__`, etc.


In [ ]:
from dataclasses import dataclass, field
from typing import Optional
import uuid

@dataclass
class ToolDefinition:
    name: str
    description: str
    parameters: dict
    required: list[str] = field(default_factory=list)

@dataclass
class AgentConfig:
    model: str = "claude-opus-4-6"
    temperature: float = 0.0
    max_tokens: int = 4096
    system_prompt: str = "You are a helpful AI agent."
    tools: list[ToolDefinition] = field(default_factory=list)
    max_iterations: int = 10

@dataclass
class AgentRun:
    run_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    config: AgentConfig = field(default_factory=AgentConfig)
    messages: list[dict] = field(default_factory=list)
    iteration: int = 0
    finished: bool = False
    error: Optional[str] = None

    def add_message(self, role: str, content: str) -> None:
        self.messages.append({"role": role, "content": content})
        self.iteration += 1

# Create a sample run
web_search_tool = ToolDefinition(
    name="web_search",
    description="Search the web for current information",
    parameters={"query": {"type": "string", "description": "Search query"}},
    required=["query"]
)

config = AgentConfig(model="claude-opus-4-6", tools=[web_search_tool])
run = AgentRun(config=config)
run.add_message("user", "What is the weather today?")

print(f"Run ID: {run.run_id[:8]}...")
print(f"Model: {run.config.model}")
print(f"Iteration: {run.iteration}")
print(f"Messages: {run.messages}")

### 1.4 Context Managers

Context managers (`with` statements) are essential for managing agent resources: tracing sessions, database connections, temporary API clients, and span tracking.


In [ ]:
from contextlib import contextmanager, asynccontextmanager
import time
from typing import Generator

# Synchronous context manager: trace a span
@contextmanager
def trace_span(name: str) -> Generator[dict, None, None]:
    span = {"name": name, "start": time.perf_counter(), "events": []}
    print(f"[TRACE] START: {name}")
    try:
        yield span
    except Exception as e:
        span["error"] = str(e)
        print(f"[TRACE] ERROR in {name}: {e}")
        raise
    finally:
        span["duration_ms"] = (time.perf_counter() - span["start"]) * 1000
        print(f"[TRACE] END: {name} ({span['duration_ms']:.1f}ms)")

# Async context manager: manage an LLM client session
@asynccontextmanager
async def llm_session(api_key: str):
    print(f"[SESSION] Opening LLM session with key ...{api_key[-4:]}")
    session_data = {"api_key": api_key, "calls": 0}
    try:
        yield session_data
    finally:
        print(f"[SESSION] Closed. Total calls: {session_data['calls']}")

# Usage
with trace_span("agent-reasoning-step") as span:
    span["events"].append("retrieved context")
    time.sleep(0.05)
    span["events"].append("called LLM")

print(f"Span events: {span['events']}")

async def demo_session():
    async with llm_session(api_key="sk-demo-key-1234") as session:
        session["calls"] += 1
        await asyncio.sleep(0.01)
        session["calls"] += 1

await demo_session()

### 1.5 Generators & Streaming

Generators are the natural Python primitive for streaming LLM output token-by-token. They allow you to process tokens as they arrive without buffering the entire response.


In [ ]:
import asyncio
from typing import AsyncGenerator

# Simulate streaming LLM output
async def stream_llm(prompt: str) -> AsyncGenerator[str, None]:
    """Yields tokens one at a time, simulating a streaming LLM response."""
    response = f"The answer to '{prompt}' is: streaming works great for agents!"
    for token in response.split():
        await asyncio.sleep(0.05)  # simulate token latency
        yield token + " "

# Consumer that collects and prints tokens as they arrive
async def consume_stream(prompt: str) -> str:
    full_response = ""
    print("Streaming: ", end="", flush=True)
    async for token in stream_llm(prompt):
        print(token, end="", flush=True)
        full_response += token
    print()  # newline
    return full_response.strip()

# Synchronous generator example: chunking a large document for RAG
def chunk_document(text: str, chunk_size: int = 100, overlap: int = 20):
    """Yields overlapping text chunks for RAG indexing."""
    start = 0
    while start < len(text):
        end = start + chunk_size
        yield text[start:end]
        start += chunk_size - overlap

sample_text = "A" * 250  # simulate a long document
chunks = list(chunk_document(sample_text, chunk_size=100, overlap=20))
print(f"Document length: {len(sample_text)} chars")
print(f"Chunks: {len(chunks)} (sizes: {[len(c) for c in chunks]})")

final = await consume_stream("what is RAG?")
print(f"Full response ({len(final)} chars)")

---
## 2. Bash Scripting for AI Workflows

Bash is the glue that binds AI pipelines together: running evals, deploying models, managing environment variables, and orchestrating multi-step workflows.

### 2.1 Core Bash Patterns for AI

The cells below show Bash scripts as strings and demonstrate running them from Python using `subprocess`.


In [ ]:
import subprocess
import textwrap

def run_bash(script: str, capture: bool = True) -> str:
    """Run a bash script string and return its output."""
    result = subprocess.run(
        ["bash", "-c", textwrap.dedent(script)],
        capture_output=capture,
        text=True
    )
    if result.returncode != 0:
        print(f"STDERR: {result.stderr}")
    return result.stdout.strip()

# 1. Environment variable management for AI projects
env_script = """
    export OPENAI_API_KEY="sk-placeholder"
    export ANTHROPIC_API_KEY="sk-ant-placeholder"
    export MODEL_NAME="claude-opus-4-6"

    echo "Model: $MODEL_NAME"
    echo "Keys set: $([ -n "$OPENAI_API_KEY" ] && echo yes || echo no)"
"""
print(run_bash(env_script))

# 2. Piping: count tokens in a text file (approximation)
pipe_script = """
    echo "The quick brown fox jumps over the lazy dog" | \
    tr ' ' '\n' | \
    grep -c '[a-z]'
"""
print(f"Word count (approx tokens): {run_bash(pipe_script)}")

# 3. Check if a Python package is installed
pkg_check = """
    packages=("anthropic" "openai" "langchain")
    for pkg in "${packages[@]}"; do
        if pip show "$pkg" &>/dev/null; then
            echo "INSTALLED: $pkg"
        else
            echo "MISSING:   $pkg"
        fi
    done
"""
print(run_bash(pkg_check))

In [ ]:
# Bash automation patterns for AI eval pipelines
eval_script = r"""
#!/usr/bin/env bash
# eval_pipeline.sh - Run evals and log results

set -euo pipefail  # exit on error, undefined vars, pipe failures

EVAL_DIR="/tmp/evals_$(date +%Y%m%d_%H%M%S)"
mkdir -p "$EVAL_DIR"

log() { echo "[$(date +%H:%M:%S)] $*" | tee -a "$EVAL_DIR/run.log"; }

log "Starting eval pipeline"
log "Output dir: $EVAL_DIR"

# Simulate running evals
for dataset in "mmlu" "humaneval" "gsm8k"; do
    log "Running eval: $dataset"
    score=$((RANDOM % 40 + 60))  # random score 60-100
    echo "{\"dataset\": \"$dataset\", \"score\": $score}" >> "$EVAL_DIR/results.jsonl"
done

log "Eval complete. Results:"
cat "$EVAL_DIR/results.jsonl"
"""

print(run_bash(eval_script))

# Process management: run a background job and capture its PID
bg_script = """
    sleep 5 &
    BG_PID=$!
    echo "Started background job PID: $BG_PID"
    echo "Job is running: $(kill -0 $BG_PID 2>/dev/null && echo yes || echo no)"
    kill $BG_PID 2>/dev/null
    echo "Job killed."
"""
print(run_bash(bg_script))

### 2.2 Useful Bash Patterns Reference

```bash
# ---- Idiomatic AI workflow patterns ----

# Run Python agent with timeout
timeout 300 python agent.py --task "summarize" || echo "Agent timed out"

# Parallel evals across multiple models
for model in gpt-4o claude-3-5-sonnet gemini-1.5-pro; do
    python eval.py --model $model &
done
wait  # wait for all background jobs

# Watch GPU memory during training
watch -n 1 nvidia-smi --query-gpu=memory.used --format=csv,noheader

# Tail logs from a running agent
tail -f agent.log | grep -E "(ERROR|WARN|tool_call)"

# Count JSONL records in a dataset
wc -l < dataset.jsonl

# Pretty-print JSON from agent output
cat output.json | python -m json.tool

# Extract all tool calls from a log
grep 'tool_call' agent.log | jq '.tool_name' | sort | uniq -c | sort -rn
```


---
## 3. TypeScript Patterns for AI Apps

TypeScript is widely used for AI-powered web apps, API servers (Node/Bun), and SDKs. Strong typing prevents runtime errors in complex agent response handling.

The following cells show TypeScript code as strings with commentary — useful as reference when building Node.js AI backends.


In [ ]:
# TypeScript code shown as Python string literals for study/reference

ts_types_example = '''
// ============================================================
// TypeScript: Core types and interfaces for AI agents
// ============================================================

// Discriminated union for agent output
type AgentOutput =
  | { type: "text"; content: string }
  | { type: "tool_use"; toolName: string; input: Record<string, unknown> }
  | { type: "error"; message: string; code: number };

// Interface for a tool definition (matches Anthropic/OpenAI schema)
interface ToolDefinition {
  name: string;
  description: string;
  inputSchema: {
    type: "object";
    properties: Record<string, { type: string; description: string }>;
    required: string[];
  };
}

// Generic result type (avoids throwing exceptions)
type Result<T, E = Error> =
  | { ok: true; value: T }
  | { ok: false; error: E };

// Utility: wrap an async function into a Result
async function safeCall<T>(
  fn: () => Promise<T>
): Promise<Result<T>> {
  try {
    return { ok: true, value: await fn() };
  } catch (err) {
    return { ok: false, error: err instanceof Error ? err : new Error(String(err)) };
  }
}

// Agent config with defaults using satisfies
const defaultConfig = {
  model: "claude-opus-4-6",
  temperature: 0,
  maxTokens: 4096,
  tools: [] as ToolDefinition[],
} satisfies AgentConfig;
'''

print(ts_types_example)

In [ ]:
ts_async_example = '''
// ============================================================
// TypeScript: Async patterns for AI apps
// ============================================================

import Anthropic from "@anthropic-ai/sdk";

const client = new Anthropic();

// 1. Basic async/await call
async function callClaude(prompt: string): Promise<string> {
  const message = await client.messages.create({
    model: "claude-opus-4-6",
    max_tokens: 1024,
    messages: [{ role: "user", content: prompt }],
  });
  const block = message.content[0];
  return block.type === "text" ? block.text : "";
}

// 2. Streaming response
async function streamClaude(prompt: string): Promise<void> {
  const stream = client.messages.stream({
    model: "claude-opus-4-6",
    max_tokens: 1024,
    messages: [{ role: "user", content: prompt }],
  });

  for await (const event of stream) {
    if (
      event.type === "content_block_delta" &&
      event.delta.type === "text_delta"
    ) {
      process.stdout.write(event.delta.text);
    }
  }
}

// 3. Parallel calls with Promise.all
async function parallelCalls(prompts: string[]): Promise<string[]> {
  return Promise.all(prompts.map(callClaude));
}

// 4. Race with timeout
function withTimeout<T>(promise: Promise<T>, ms: number): Promise<T> {
  return Promise.race([
    promise,
    new Promise<never>((_, reject) =>
      setTimeout(() => reject(new Error(`Timeout after ${ms}ms`)), ms)
    ),
  ]);
}
'''

print(ts_async_example)

In [ ]:
ts_zod_example = '''
// ============================================================
// TypeScript: Runtime validation with Zod (essential for LLM outputs)
// ============================================================

import { z } from "zod";

// Define the expected shape of an LLM JSON response
const AgentActionSchema = z.object({
  action: z.enum(["search", "calculate", "respond", "handoff"]),
  reasoning: z.string().min(1),
  parameters: z.record(z.string(), z.unknown()).optional(),
  confidence: z.number().min(0).max(1),
});

type AgentAction = z.infer<typeof AgentActionSchema>;

// Parse and validate LLM JSON output safely
function parseAgentAction(raw: unknown): Result<AgentAction> {
  const result = AgentActionSchema.safeParse(raw);
  if (result.success) {
    return { ok: true, value: result.data };
  }
  return {
    ok: false,
    error: new Error(result.error.issues.map(i => i.message).join(", ")),
  };
}

// Example: parse LLM response
const llmResponse = JSON.parse(
  `{"action":"search","reasoning":"User asked about weather","confidence":0.9}`
);
const parsed = parseAgentAction(llmResponse);
if (parsed.ok) {
  console.log("Action:", parsed.value.action);
} else {
  console.error("Invalid LLM output:", parsed.error.message);
}
'''

print(ts_zod_example)

---
## 4. Data Structures & Algorithms for Agents

Agents rely on specific data structures for their core operations:
- **Queues / Deques** — message history, task queues, BFS
- **Priority Queues (heaps)** — scheduling tasks by priority, A* search
- **Graphs** — tool dependency graphs, knowledge graphs, agent DAGs
- **Tries / Hash maps** — fast tool lookup, memory indexing


In [ ]:
from collections import deque
from dataclasses import dataclass, field
from typing import Optional

# ---- Task Queue for an Agent ----
@dataclass
class Task:
    id: str
    description: str
    status: str = "pending"  # pending | running | done | failed

class AgentTaskQueue:
    """FIFO task queue backed by a deque."""
    def __init__(self):
        self._queue: deque[Task] = deque()
        self._completed: list[Task] = []

    def enqueue(self, task: Task) -> None:
        self._queue.append(task)

    def dequeue(self) -> Optional[Task]:
        if self._queue:
            task = self._queue.popleft()
            task.status = "running"
            return task
        return None

    def complete(self, task: Task) -> None:
        task.status = "done"
        self._completed.append(task)

    def __len__(self): return len(self._queue)

    def __repr__(self):
        return f"AgentTaskQueue(pending={len(self._queue)}, done={len(self._completed)})"

# Demo
queue = AgentTaskQueue()
for i, desc in enumerate(["fetch docs", "summarize", "write report", "send email"]):
    queue.enqueue(Task(id=f"task-{i}", description=desc))

print(f"Initial: {queue}")
while task := queue.dequeue():
    print(f"  Running: {task.description}")
    queue.complete(task)
print(f"Final: {queue}")

In [ ]:
import heapq
from dataclasses import dataclass, field

@dataclass(order=True)
class PrioritizedTask:
    priority: int          # lower = higher priority
    task_id: str = field(compare=False)
    description: str = field(compare=False)

class PriorityTaskQueue:
    """Min-heap based priority queue for agent tasks."""
    def __init__(self):
        self._heap: list[PrioritizedTask] = []

    def push(self, task: PrioritizedTask) -> None:
        heapq.heappush(self._heap, task)

    def pop(self) -> Optional[PrioritizedTask]:
        return heapq.heappop(self._heap) if self._heap else None

    def peek(self) -> Optional[PrioritizedTask]:
        return self._heap[0] if self._heap else None

    def __len__(self): return len(self._heap)

# Demo: scheduling agent tasks by urgency
pq = PriorityTaskQueue()
tasks = [
    PrioritizedTask(priority=3, task_id="t1", description="Write final report"),
    PrioritizedTask(priority=1, task_id="t2", description="Handle user interruption (URGENT)"),
    PrioritizedTask(priority=2, task_id="t3", description="Summarize retrieved docs"),
    PrioritizedTask(priority=1, task_id="t4", description="Retry failed API call (URGENT)"),
]
for t in tasks:
    pq.push(t)

print("Processing tasks in priority order:")
while task := pq.pop():
    print(f"  [P{task.priority}] {task.description}")

In [ ]:
from collections import defaultdict, deque
from typing import Optional

class DirectedGraph:
    """Adjacency-list directed graph — models agent tool dependency DAGs."""
    def __init__(self):
        self.graph: dict[str, list[str]] = defaultdict(list)
        self.nodes: set[str] = set()

    def add_edge(self, src: str, dst: str) -> None:
        self.graph[src].append(dst)
        self.nodes.update([src, dst])

    def bfs(self, start: str) -> list[str]:
        """Breadth-first traversal — useful for level-by-level agent planning."""
        visited, order = set(), []
        queue = deque([start])
        while queue:
            node = queue.popleft()
            if node in visited:
                continue
            visited.add(node)
            order.append(node)
            queue.extend(self.graph[node])
        return order

    def topological_sort(self) -> list[str]:
        """Kahn's algorithm — determines safe execution order for a tool DAG."""
        in_degree = {n: 0 for n in self.nodes}
        for src in self.graph:
            for dst in self.graph[src]:
                in_degree[dst] += 1

        queue = deque(n for n in self.nodes if in_degree[n] == 0)
        order = []
        while queue:
            node = queue.popleft()
            order.append(node)
            for neighbor in self.graph[node]:
                in_degree[neighbor] -= 1
                if in_degree[neighbor] == 0:
                    queue.append(neighbor)
        return order if len(order) == len(self.nodes) else []  # empty = cycle

# Model a tool dependency DAG
# search_web --> extract_facts --> synthesize_answer --> format_output
#                                  fetch_db ---------->/
dag = DirectedGraph()
dag.add_edge("search_web", "extract_facts")
dag.add_edge("fetch_db", "synthesize_answer")
dag.add_edge("extract_facts", "synthesize_answer")
dag.add_edge("synthesize_answer", "format_output")

print("BFS from search_web:", dag.bfs("search_web"))
print("Topological order:", dag.topological_sort())

In [ ]:
# Sliding window: token budget management for context windows
from collections import deque

def trim_to_token_budget(messages: list[dict], max_tokens: int, tokens_per_msg: int = 100) -> list[dict]:
    """
    Keep the most recent messages that fit within a token budget.
    Always preserve the system message (index 0).
    """
    if not messages:
        return messages

    system = messages[0] if messages[0]["role"] == "system" else None
    history = messages[1:] if system else messages[:]

    budget = max_tokens - (tokens_per_msg if system else 0)
    window = deque()
    used = 0

    for msg in reversed(history):
        cost = len(msg["content"].split()) + 4  # rough token estimate
        if used + cost > budget:
            break
        window.appendleft(msg)
        used += cost

    return ([system] if system else []) + list(window)

# Test with a growing conversation
msgs = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "What is Python?"},
    {"role": "assistant", "content": "Python is a high-level programming language."},
    {"role": "user",   "content": "What is asyncio?"},
    {"role": "assistant", "content": "asyncio is Python's async I/O framework."},
    {"role": "user",   "content": "How do I use await?"},
]

trimmed = trim_to_token_budget(msgs, max_tokens=50)
print(f"Original: {len(msgs)} messages")
print(f"Trimmed:  {len(trimmed)} messages")
for m in trimmed:
    print(f"  [{m['role']}]: {m['content'][:50]}")

---
## 5. Git & Version Control for AI Projects

AI projects have unique version control needs:
- **Model weights** are large binary files (use Git LFS or DVC)
- **Eval results** should be tracked alongside code changes
- **Prompts** are code — they should be versioned, reviewed, and tested
- **Branching strategies** must accommodate both research (experimental) and production (stable) workflows


In [ ]:
# Git workflows for AI projects — demonstrated via subprocess
import subprocess
import os
import tempfile

def git(args: list[str], cwd: str) -> str:
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

# Create a temp repo to demonstrate
tmpdir = tempfile.mkdtemp(prefix="ai_project_")

# Initialize repo
print(git(["init"], tmpdir))
print(git(["config", "user.email", "agent@example.com"], tmpdir))
print(git(["config", "user.name", "Agent Dev"], tmpdir))

# Create a .gitignore for AI projects
gitignore = """# AI project .gitignore
*.pkl
*.pt
*.bin
models/weights/
.env
*.env*
__pycache__/
.venv/
*.jsonl.bak
wandb/
mlruns/
"""
with open(os.path.join(tmpdir, ".gitignore"), "w") as f:
    f.write(gitignore)

with open(os.path.join(tmpdir, "README.md"), "w") as f:
    f.write("# My AI Agent Project\n")

print(git(["add", "."], tmpdir))
print(git(["commit", "-m", "chore: initial project setup with .gitignore"], tmpdir))
print(git(["log", "--oneline"], tmpdir))

In [ ]:
# Branching strategies for AI projects

# Create feature branch for a new prompt
print(git(["checkout", "-b", "feature/improved-system-prompt"], tmpdir))

# Write a versioned prompt file
import json, os

prompts = {
    "version": "1.1.0",
    "system": "You are a highly capable AI agent. Think step by step.",
    "tools_intro": "You have access to the following tools:",
    "format": "Always respond with valid JSON."
}

prompts_path = os.path.join(tmpdir, "prompts.json")
with open(prompts_path, "w") as f:
    json.dump(prompts, f, indent=2)

print(git(["add", "prompts.json"], tmpdir))
print(git(["commit", "-m", "feat: add versioned system prompt v1.1.0"], tmpdir))

# Go back to main and merge
print(git(["checkout", "master"], tmpdir))  # or main
print(git(["merge", "--no-ff", "feature/improved-system-prompt",
           "-m", "merge: improved system prompt"], tmpdir))

# Tag a model/prompt version
print(git(["tag", "-a", "prompt-v1.1.0", "-m", "Prompt version 1.1.0 — improved chain-of-thought"], tmpdir))

print("\nCommit history:")
print(git(["log", "--oneline", "--graph", "--all"], tmpdir))

print("\nTags:")
print(git(["tag", "-l", "-n1"], tmpdir))

In [ ]:
# Git best practices reference for AI projects

git_best_practices = {
    "Branch naming": [
        "feature/agent-memory-v2",
        "experiment/gpt4o-vs-claude-eval",
        "fix/tool-call-parsing",
        "release/v1.2.0",
    ],
    "Commit message prefixes": [
        "feat: new capability",
        "fix: bug in tool parser",
        "eval: updated benchmark results",
        "prompt: revised system prompt",
        "chore: update dependencies",
        "docs: add architecture diagram",
    ],
    "Large file strategies": [
        "Git LFS for model checkpoints (.bin, .pt, .gguf)",
        "DVC (Data Version Control) for datasets and eval results",
        "Store embeddings in vector DBs, not in git",
        "Use .gitignore to exclude .env and secrets",
    ],
    "Useful git aliases for AI work": [
        "git log --oneline --graph --all",
        "git diff HEAD~1 prompts.json  # see prompt changes",
        "git bisect  # find which commit broke eval scores",
        "git tag -a model-v2.1 -m 'Fine-tuned on v2 dataset'",
    ]
}

for category, items in git_best_practices.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  - {item}")

---
## 6. Testing with PyTest

Testing AI agents requires special techniques because:
1. LLM calls are **non-deterministic** — you must mock them
2. Agents have **side effects** — tool calls, API calls, state mutations
3. **Eval tests** are separate from unit tests — measure quality, not correctness

### 6.1 Mocking LLM Calls


In [ ]:
# Show test structure as strings (these would be in test_agent.py)
# We'll also run them in-process using pytest programmatically

test_mocking_code = '''
# test_agent.py
import pytest
from unittest.mock import AsyncMock, patch, MagicMock

# The agent module under test
# from agent import Agent, run_agent_step

# ---- Fixtures ----

@pytest.fixture
def mock_anthropic_client():
    """Returns a mock Anthropic client that returns a predictable response."""
    client = MagicMock()
    # Mock the messages.create coroutine
    mock_message = MagicMock()
    mock_message.content = [MagicMock(type="text", text="Paris")]
    mock_message.stop_reason = "end_turn"
    client.messages.create = AsyncMock(return_value=mock_message)
    return client

@pytest.fixture
def sample_messages():
    return [
        {"role": "user", "content": "What is the capital of France?"}
    ]

# ---- Unit tests ----

@pytest.mark.asyncio
async def test_agent_returns_text_response(mock_anthropic_client, sample_messages):
    """Agent should return a text string when LLM returns end_turn."""
    # result = await run_agent_step(mock_anthropic_client, sample_messages)
    # assert result == "Paris"
    response = await mock_anthropic_client.messages.create(messages=sample_messages)
    assert response.content[0].text == "Paris"
    assert response.stop_reason == "end_turn"

@pytest.mark.asyncio
async def test_llm_called_with_correct_model(mock_anthropic_client, sample_messages):
    """Agent must call the LLM with the configured model name."""
    await mock_anthropic_client.messages.create(
        model="claude-opus-4-6",
        max_tokens=4096,
        messages=sample_messages
    )
    mock_anthropic_client.messages.create.assert_called_once_with(
        model="claude-opus-4-6",
        max_tokens=4096,
        messages=sample_messages
    )
'''

print(test_mocking_code)

In [ ]:
# Run actual pytest tests in-process to demonstrate
import pytest
from unittest.mock import AsyncMock, MagicMock, patch
import asyncio

# ---- Functions under test (inline for demo) ----

async def call_llm_for_test(client, messages: list[dict], model: str = "claude-opus-4-6") -> str:
    response = await client.messages.create(
        model=model,
        max_tokens=1024,
        messages=messages
    )
    block = response.content[0]
    return block.text if block.type == "text" else ""

def parse_tool_call(response_text: str) -> dict | None:
    """Parse a JSON tool call from an LLM text response."""
    import json
    try:
        data = json.loads(response_text)
        if "tool" in data:
            return data
    except json.JSONDecodeError:
        pass
    return None

# ---- Tests ----

async def test_call_llm_returns_text():
    mock_client = MagicMock()
    mock_response = MagicMock()
    mock_response.content = [MagicMock(type="text", text="The capital is Paris.")]
    mock_client.messages.create = AsyncMock(return_value=mock_response)

    result = await call_llm_for_test(mock_client, [{"role": "user", "content": "Capital of France?"}])
    assert result == "The capital is Paris."
    print("  PASS: test_call_llm_returns_text")

def test_parse_tool_call_valid():
    raw = '{"tool": "web_search", "query": "AI news"}'
    result = parse_tool_call(raw)
    assert result is not None
    assert result["tool"] == "web_search"
    print("  PASS: test_parse_tool_call_valid")

def test_parse_tool_call_invalid():
    result = parse_tool_call("Just a plain text response.")
    assert result is None
    print("  PASS: test_parse_tool_call_invalid")

def test_parse_tool_call_json_no_tool_key():
    result = parse_tool_call('{"answer": "42"}')
    assert result is None
    print("  PASS: test_parse_tool_call_json_no_tool_key")

# Run all tests
print("Running tests...")
await test_call_llm_returns_text()
test_parse_tool_call_valid()
test_parse_tool_call_invalid()
test_parse_tool_call_json_no_tool_key()
print("All tests passed.")

In [ ]:
# Advanced pytest patterns for agent testing

pytest_advanced = '''
# conftest.py — shared fixtures for agent tests
import pytest
from unittest.mock import AsyncMock, MagicMock

# ---- Parametrized tests: test across multiple models ----

@pytest.mark.parametrize("model", [
    "claude-opus-4-6",
    "gpt-4o",
    "gemini-1.5-pro",
])
def test_agent_config_model_name(model):
    config = AgentConfig(model=model)
    assert config.model == model

# ---- Fixture with scope ----

@pytest.fixture(scope="session")
def embedding_model():
    """Load once per test session — expensive setup."""
    return FakeEmbeddingModel()

# ---- Mocking with patch ----

@patch("agent.anthropic_client.messages.create")
async def test_agent_handles_rate_limit(mock_create):
    from anthropic import RateLimitError
    mock_create.side_effect = RateLimitError("Rate limited", response=MagicMock(), body={})
    with pytest.raises(RateLimitError):
        await run_agent("Hello")

# ---- Testing tool execution ----

async def test_tool_registry_calls_correct_function():
    mock_search = AsyncMock(return_value="Search results")
    registry = ToolRegistry({"web_search": mock_search})

    result = await registry.execute("web_search", {"query": "AI news"})

    mock_search.assert_called_once_with(query="AI news")
    assert result == "Search results"

# ---- Snapshot testing for prompts ----

def test_system_prompt_unchanged(snapshot):
    """Fails if the system prompt changes unexpectedly — prompts are code!"""
    from prompts import SYSTEM_PROMPT
    snapshot.assert_match(SYSTEM_PROMPT, "system_prompt.txt")
'''

print(pytest_advanced)

In [ ]:
# Eval testing: measuring agent quality (separate from unit tests)

from dataclasses import dataclass
from typing import Callable

@dataclass
class EvalCase:
    input: str
    expected_contains: list[str]  # keywords that must appear in the response
    expected_not_contains: list[str] = None

    def __post_init__(self):
        if self.expected_not_contains is None:
            self.expected_not_contains = []

def evaluate_response(case: EvalCase, response: str) -> dict:
    """Score a single eval case."""
    response_lower = response.lower()
    hits = sum(1 for kw in case.expected_contains if kw.lower() in response_lower)
    misses = sum(1 for kw in case.expected_not_contains if kw.lower() in response_lower)
    total_checks = len(case.expected_contains) + len(case.expected_not_contains)
    score = (hits + len(case.expected_not_contains) - misses) / total_checks if total_checks else 1.0
    return {
        "input": case.input,
        "score": round(score, 3),
        "hits": hits,
        "misses": misses,
        "response_preview": response[:80]
    }

# Example eval suite
eval_cases = [
    EvalCase(
        input="What is RAG?",
        expected_contains=["retrieval", "generation", "documents"],
        expected_not_contains=["I don't know"]
    ),
    EvalCase(
        input="How do agents use tools?",
        expected_contains=["function", "call", "API"],
        expected_not_contains=["cannot", "unable"]
    ),
]

# Simulated agent responses
mock_responses = [
    "RAG stands for Retrieval-Augmented Generation. It retrieves relevant documents and uses them for generation.",
    "Agents use tools by making function calls to external APIs and processing the results."
]

print("Eval Results:")
total_score = 0
for case, resp in zip(eval_cases, mock_responses):
    result = evaluate_response(case, resp)
    total_score += result["score"]
    print(f"  Q: {result['input'][:40]}...")
    print(f"     Score: {result['score']:.1%} | hits={result['hits']} misses={result['misses']}")

print(f"\nOverall score: {total_score / len(eval_cases):.1%}")

---
## 7. Putting It All Together: A Mini Agent Skeleton

The following cell combines all the concepts above into a minimal but realistic agent implementation skeleton.


In [ ]:
import asyncio
import json
import uuid
from dataclasses import dataclass, field
from typing import AsyncGenerator, Callable, Awaitable, Optional
from contextlib import asynccontextmanager

# ---- Types ----
ToolHandler = Callable[[dict], Awaitable[str]]

@dataclass
class Tool:
    name: str
    description: str
    handler: ToolHandler

@dataclass
class AgentState:
    run_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    messages: list[dict] = field(default_factory=list)
    iterations: int = 0
    done: bool = False

# ---- Tool registry ----
class ToolRegistry:
    def __init__(self, tools: list[Tool]):
        self._tools = {t.name: t for t in tools}

    async def execute(self, name: str, args: dict) -> str:
        if name not in self._tools:
            return f"ERROR: unknown tool '{name}'"
        try:
            return await self._tools[name].handler(args)
        except Exception as e:
            return f"ERROR: {e}"

    def descriptions(self) -> list[dict]:
        return [{"name": t.name, "description": t.description} for t in self._tools.values()]

# ---- Simulated LLM ----
async def fake_llm(messages: list[dict], tools: list[dict]) -> dict:
    """Returns a tool call on first turn, then a final answer."""
    user_msgs = [m for m in messages if m["role"] == "user"]
    if len(user_msgs) == 1:  # first turn: use a tool
        return {"type": "tool_call", "tool": "web_search", "args": {"query": "AI agents 2025"}}
    else:  # second turn: final answer
        return {"type": "text", "content": "AI agents are autonomous systems that use LLMs and tools."}

# ---- Agent loop ----
async def run_agent(user_input: str, tools: list[Tool], max_iter: int = 5) -> str:
    state = AgentState()
    registry = ToolRegistry(tools)
    state.messages.append({"role": "user", "content": user_input})

    while not state.done and state.iterations < max_iter:
        state.iterations += 1
        response = await fake_llm(state.messages, registry.descriptions())

        if response["type"] == "tool_call":
            tool_result = await registry.execute(response["tool"], response["args"])
            state.messages.append({"role": "assistant", "content": f"[tool call: {response['tool']}]"})
            state.messages.append({"role": "tool", "content": tool_result})
            print(f"  [iter {state.iterations}] Tool: {response['tool']} -> {tool_result[:50]}")
        elif response["type"] == "text":
            state.done = True
            state.messages.append({"role": "assistant", "content": response["content"]})
            print(f"  [iter {state.iterations}] Final answer: {response['content']}")
            return response["content"]

    return "Max iterations reached"

# ---- Define tools ----
async def web_search(args: dict) -> str:
    query = args.get("query", "")
    await asyncio.sleep(0.1)  # simulate latency
    return f"Search results for '{query}': [doc1, doc2, doc3]"

tools = [Tool(name="web_search", description="Search the web", handler=web_search)]

# ---- Run ----
print("Starting agent...")
final = await run_agent("What are AI agents?", tools)
print(f"\nFinal response: {final}")

---
## 8. Quick Reference Cheatsheets


In [ ]:
cheatsheets = {
    "Python Async Patterns": {
        "Single call":         "result = await coroutine()",
        "Parallel calls":      "r1, r2 = await asyncio.gather(c1(), c2())",
        "With timeout":        "result = await asyncio.wait_for(coro(), timeout=5)",
        "Async context":       "async with resource() as r: ...",
        "Async generator":     "async for token in stream_llm(prompt): ...",
        "Run from sync":       "asyncio.run(main())",
    },
    "Dataclass Patterns": {
        "Basic":               "@dataclass\nclass Foo: x: int",
        "Default factory":     "items: list = field(default_factory=list)",
        "Post init":           "def __post_init__(self): ...",
        "Frozen (immutable)":  "@dataclass(frozen=True)",
    },
    "Git AI Workflow": {
        "Create branch":       "git checkout -b feature/my-agent",
        "Tag a version":       "git tag -a v1.2.0 -m 'description'",
        "Find regression":     "git bisect start && git bisect bad && git bisect good <sha>",
        "Track large files":   "git lfs track '*.bin' && git add .gitattributes",
    },
    "PyTest Agent Testing": {
        "Mock async call":     "mock.create = AsyncMock(return_value=mock_resp)",
        "Assert called":       "mock.create.assert_called_once_with(...)",
        "Parametrize":         "@pytest.mark.parametrize('x', [1, 2, 3])",
        "Raises":              "with pytest.raises(ValueError): ...",
        "Async test":          "@pytest.mark.asyncio async def test_(): ...",
    },
    "Data Structures for Agents": {
        "FIFO task queue":     "from collections import deque; q = deque()",
        "Priority queue":      "import heapq; heapq.heappush(h, (priority, item))",
        "Adjacency list":      "graph = defaultdict(list)",
        "Topological sort":    "Use Kahn's algorithm for DAG execution order",
        "Token window":        "Use deque with maxlen for sliding window",
    }
}

for section, items in cheatsheets.items():
    print(f"\n{'='*60}")
    print(f" {section}")
    print(f"{'='*60}")
    for key, val in items.items():
        print(f"  {key:<30} {val}")

In [ ]:
further_reading = [
    {
        "topic": "Python asyncio",
        "resources": [
            "Python docs: asyncio — https://docs.python.org/3/library/asyncio.html",
            "'Python Concurrency with asyncio' by Matthew Fowler (book)",
        ]
    },
    {
        "topic": "TypeScript for Node.js AI",
        "resources": [
            "Anthropic TypeScript SDK — https://github.com/anthropics/anthropic-sdk-typescript",
            "Zod schema validation — https://zod.dev",
        ]
    },
    {
        "topic": "Testing AI systems",
        "resources": [
            "pytest-asyncio — https://pytest-asyncio.readthedocs.io",
            "Braintrust eval framework — https://braintrust.dev",
            "promptfoo — https://promptfoo.dev",
        ]
    },
    {
        "topic": "Model versioning & datasets",
        "resources": [
            "DVC (Data Version Control) — https://dvc.org",
            "Git LFS — https://git-lfs.github.com",
            "Weights & Biases Artifacts — https://wandb.ai/site/artifacts",
        ]
    },
    {
        "topic": "Agentic frameworks to study",
        "resources": [
            "LangGraph — https://langchain-ai.github.io/langgraph",
            "Anthropic agent patterns — https://github.com/anthropics/anthropic-cookbook",
            "OpenAI Swarm — https://github.com/openai/swarm",
        ]
    },
]

print("Further Reading & Resources")
print("="*50)
for entry in further_reading:
    print(f"\n{entry['topic']}:")
    for r in entry["resources"]:
        print(f"  - {r}")